### ID3 Decision Tree Implementation

In [94]:
import math
from collections import Counter
import pandas as pd
import random

In [95]:
def get_entropy(labels):
    if not labels:
        return 0.0
    else:
        entropy = 0.0
        counts = Counter(labels)
        total  = len(labels)
        for count in counts.values():
            if count > 0:
                entropy -= (count / total) * math.log2(count / total)
        return entropy
    
def information_gain(data, labels, attribute_idx):
    ''' Returns the information gain of a certain attirbute'''
    h_c = get_entropy(labels)
    # Group labels by attribute value
    subsets = {}
    for i, row in enumerate(data):
        key = row[attribute_idx]
        subsets.setdefault(key, []).append(labels[i])
    total = len(labels)
    conditional_entropy = 0.0
    for subset_labels in subsets.values():
        probability = len(subset_labels) / total
        conditional_entropy += probability * get_entropy(subset_labels)
    gain = h_c - conditional_entropy
    return gain

In [96]:
class DecisionTreeNode:
    def __init__(self, is_leaf=False, label=None, feature_idx=None):
        self.is_leaf      = is_leaf
        self.label        = label
        self.feature_idx  = feature_idx
        self.children     = {}

def id3(data, labels, feature_indices, depth=0, max_depth=None):
    # Base cases
    # 1 - All the examples in the node have the same label: it is a leaf
    if (len(set(labels))) == 1: 
        return DecisionTreeNode(is_leaf=True, label=labels[0])
    # 2 - No attributes for splitting 
    if not feature_indices:
        majority = Counter(labels).most_common(1)[0][0]
        return DecisionTreeNode(is_leaf=True, label=majority)
    
    # Choosing the attribute with the highest information gain
    best_idx = None
    best_gain = 0.0
    for idx in feature_indices:
        gain = information_gain(data, labels, idx)
        if gain > best_gain:
            best_gain = gain
            best_idx = idx
    
    # If no best index found, use majority label
    if best_idx is None:
        majority = Counter(labels).most_common(1)[0][0]
        return DecisionTreeNode(is_leaf=True, label=majority)
    
    node = DecisionTreeNode(feature_idx = best_idx)
    values = set(row[best_idx] for row in data)
    remaining = [idx for idx in feature_indices if idx != best_idx]
    
    # Construct subtree
    for val in values:
        sub_data   = [data[i] for i in range(len(data)) if data[i][best_idx] == val]
        sub_labels = [labels[i] for i in range(len(labels)) if data[i][best_idx] == val]
        # Recursive call
        node.children[val] = id3(sub_data, sub_labels, remaining, depth + 1, max_depth)
    return node

def predict(tree, value):
    node = tree
    while not node.is_leaf:
        if node.feature_idx is None:
            return None
        val = value[node.feature_idx]
        if val not in node.children:
            return None  # valor desconhecido
        node = node.children[val]
    return node.label

def accuracy(tree, data, labels):
    total = len(labels)
    correct = 0
    for i, ex in enumerate(data):
        if predict(tree, ex) == labels[i]:
            correct += 1
    if not labels:
        return 0.0
    else:
        return correct / total

Test the tree on the iris dataset

In [97]:
df = pd.read_csv('iris.csv')
features = df.columns[1:-1]
label = df.columns[-1]

def learn_bins(data, n_bins = 3):
    '''Learn bin boundaries from data'''
    min_val = min(data)
    max_val = max(data)
    bin_size = (max_val - min_val) / n_bins
    return [min_val + bin_size, min_val + 2 * bin_size]

def discretize_value(value, bins):
    '''Discretize a single value using predefined bins'''
    if value <= bins[0]:
        return "low"
    elif value <= bins[1]:
        return "medium"
    else:
        return "high"

In [ ]:
# Train and test splitting with separate discretization
random.seed(1234)
indices = list(range(len(df)))
random.shuffle(indices)

split = int(0.8 * len(indices))
train_idx = indices[:split]
test_idx = indices[split:]

# Get raw train and test data
train_raw = df.iloc[train_idx]
test_raw = df.iloc[test_idx]

# Learn bins from training data
train_bins = {}
for column in features:
    train_bins[column] = learn_bins(train_raw[column].values)

# Learn bins from test data
test_bins = {}
for column in features:
    test_bins[column] = learn_bins(test_raw[column].values)

# Discretize train_data using train bins
train_data = []
for i in train_idx:
    row = []
    for column in features:
        value = df.loc[i, column]
        discretized_value = discretize_value(value, train_bins[column])
        row.append(discretized_value)
    train_data.append(row)

# Discretize test_data using test bins
test_data = []
for i in test_idx:
    row = []
    for column in features:
        value = df.loc[i, column]
        discretized_value = discretize_value(value, test_bins[column])
        row.append(discretized_value)
    test_data.append(row)

train_labels = [df.loc[i, label] for i in train_idx]
test_labels = [df.loc[i, label] for i in test_idx]

feature_indices = list(range(len(features)))
my_tree = id3(train_data, train_labels, feature_indices)

train_accuracy = accuracy(my_tree, train_data, train_labels)
test_accuracy = accuracy(my_tree, test_data, test_labels)

print(f'\nTrain accuracy: {train_accuracy:.1%}')
print(f'Test accuracy:  {test_accuracy:.1%}')


Train accuracy: 95.8%
Test accuracy:  90.0%
